## Comparaison du fichier 2016 et du fichier historique

In [69]:
import pandas as pd

path_2016 = "../data/raw/2016_Building_Energy_Benchmarking.csv"
path_hist = "../data/raw/Building_Energy_Benchmarking_Data,_2015-Present_20260330.csv"

df_2016 = pd.read_csv(path_2016, low_memory=False)
df_hist = pd.read_csv(path_hist, low_memory=False)

print("df_2016 :", df_2016.shape)
print("df_hist :", df_hist.shape)

df_2016 : (3376, 46)
df_hist : (34699, 46)


In [70]:
cols_2016 = set(df_2016.columns)
cols_hist = set(df_hist.columns)

common_cols = sorted(cols_2016 & cols_hist)
only_2016 = sorted(cols_2016 - cols_hist)
only_hist = sorted(cols_hist - cols_2016)

print("Colonnes communes :", len(common_cols))
print(common_cols)

print("\nColonnes uniquement dans 2016 :", len(only_2016))
print(only_2016)

print("\nColonnes uniquement dans historique :", len(only_hist))
print(only_hist)

Colonnes communes : 38
['Address', 'BuildingType', 'City', 'ComplianceStatus', 'CouncilDistrictCode', 'DataYear', 'ENERGYSTARScore', 'Electricity(kBtu)', 'Electricity(kWh)', 'GHGEmissionsIntensity', 'LargestPropertyUseType', 'LargestPropertyUseTypeGFA', 'Latitude', 'Longitude', 'NaturalGas(kBtu)', 'NaturalGas(therms)', 'Neighborhood', 'NumberofBuildings', 'NumberofFloors', 'OSEBuildingID', 'PropertyGFAParking', 'PropertyGFATotal', 'SecondLargestPropertyUseType', 'SecondLargestPropertyUseTypeGFA', 'SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)', 'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)', 'State', 'SteamUse(kBtu)', 'TaxParcelIdentificationNumber', 'ThirdLargestPropertyUseType', 'ThirdLargestPropertyUseTypeGFA', 'TotalGHGEmissions', 'YearBuilt', 'ZipCode']

Colonnes uniquement dans 2016 : 8
['Comments', 'DefaultData', 'ListOfAllPropertyUseTypes', 'Outlier', 'PrimaryPropertyType', 'PropertyGFABuilding(s)', 'PropertyName', 'YearsENERGYSTARCertifie

In [71]:
if "DataYear" in df_hist.columns:
    print(df_hist["DataYear"].value_counts().sort_index())

DataYear
2015    3175
2016    3253
2017    3310
2018    3381
2019    3456
2020    3530
2021    3578
2022    3617
2023    3687
2024    3712
Name: count, dtype: int64


In [72]:
if {"OSEBuildingID", "DataYear"}.issubset(df_hist.columns):
    print("Doublons sur (OSEBuildingID, DataYear) :", df_hist.duplicated(subset=["OSEBuildingID", "DataYear"]).sum())

Doublons sur (OSEBuildingID, DataYear) : 0


## Harmonisation des colonnes

In [73]:
rename_2016 = {
    "PrimaryPropertyType": "EPAPropertyType",
    "PropertyGFABuilding(s)": "PropertyGFABuildings",
    "PropertyName": "BuildingName"
}

df_2016_renamed = df_2016.rename(columns=rename_2016).copy()

print("df_2016_renamed :", df_2016_renamed.shape)

df_2016_renamed : (3376, 46)


In [74]:
common_after_rename = sorted(set(df_2016_renamed.columns) & set(df_hist.columns))
print("Colonnes communes après harmonisation :", len(common_after_rename))
print(common_after_rename)

Colonnes communes après harmonisation : 41
['Address', 'BuildingName', 'BuildingType', 'City', 'ComplianceStatus', 'CouncilDistrictCode', 'DataYear', 'ENERGYSTARScore', 'EPAPropertyType', 'Electricity(kBtu)', 'Electricity(kWh)', 'GHGEmissionsIntensity', 'LargestPropertyUseType', 'LargestPropertyUseTypeGFA', 'Latitude', 'Longitude', 'NaturalGas(kBtu)', 'NaturalGas(therms)', 'Neighborhood', 'NumberofBuildings', 'NumberofFloors', 'OSEBuildingID', 'PropertyGFABuildings', 'PropertyGFAParking', 'PropertyGFATotal', 'SecondLargestPropertyUseType', 'SecondLargestPropertyUseTypeGFA', 'SiteEUI(kBtu/sf)', 'SiteEUIWN(kBtu/sf)', 'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)', 'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)', 'State', 'SteamUse(kBtu)', 'TaxParcelIdentificationNumber', 'ThirdLargestPropertyUseType', 'ThirdLargestPropertyUseTypeGFA', 'TotalGHGEmissions', 'YearBuilt', 'ZipCode']


## Construction d'un dataset historique homogène

In [75]:
df_2016_common = df_2016_renamed[common_after_rename].copy()
df_hist_common = df_hist[common_after_rename].copy()

df_2016_common["source_file"] = "2016_detail"
df_hist_common["source_file"] = "2015_present"

print(df_2016_common.shape, df_hist_common.shape)

(3376, 42) (34699, 42)


In [76]:
df_all = pd.concat([df_2016_common, df_hist_common], axis=0, ignore_index=True)

print("Avant déduplication :", df_all.shape)
print("Doublons (OSEBuildingID, DataYear) :", df_all.duplicated(subset=["OSEBuildingID", "DataYear"]).sum())

Avant déduplication : (38075, 42)
Doublons (OSEBuildingID, DataYear) : 3194


In [77]:
# Priorité au fichier 2016 détaillé pour l'année 2016
df_all["source_priority"] = df_all["source_file"].map({
    "2016_detail": 0,
    "2015_present": 1
})

df_all = df_all.sort_values(["OSEBuildingID", "DataYear", "source_priority"])
df_all = df_all.drop_duplicates(subset=["OSEBuildingID", "DataYear"], keep="first").copy()

print("Après déduplication :", df_all.shape)

Après déduplication : (34881, 43)


In [78]:
df_all["DataYear"].value_counts().sort_index()

DataYear
2015    3175
2016    3435
2017    3310
2018    3381
2019    3456
2020    3530
2021    3578
2022    3617
2023    3687
2024    3712
Name: count, dtype: int64

## Filtrage métier sur le dataset historique

In [79]:
building_types_to_keep = [
    "NonResidential",
    "Nonresidential COS",
    "Nonresidential WA",
    "Campus",
    "SPS-District K-12"
]

numeric_cols_to_fix = [
    "SiteEnergyUse(kBtu)",
    "PropertyGFATotal",
    "NumberofBuildings",
    "NumberofFloors",
    "LargestPropertyUseTypeGFA",
    "SecondLargestPropertyUseTypeGFA",
    "Latitude",
    "Longitude",
    "YearBuilt",
    "DataYear"
]

In [80]:
df_hist_model = df_all.copy()

for col in numeric_cols_to_fix:
    if col in df_hist_model.columns:
        df_hist_model[col] = pd.to_numeric(df_hist_model[col], errors="coerce")

df_hist_model = df_hist_model[df_hist_model["BuildingType"].isin(building_types_to_keep)].copy()
df_hist_model = df_hist_model[df_hist_model["SiteEnergyUse(kBtu)"].notna()].copy()
df_hist_model = df_hist_model[df_hist_model["SiteEnergyUse(kBtu)"] > 0].copy()
df_hist_model = df_hist_model[df_hist_model["PropertyGFATotal"].notna()].copy()
df_hist_model = df_hist_model[df_hist_model["PropertyGFATotal"] > 0].copy()

print(df_hist_model.shape)
print(df_hist_model["DataYear"].value_counts().sort_index())

(1650, 43)
DataYear
2016    1650
Name: count, dtype: int64


In [81]:
print(df_hist_model[["SiteEnergyUse(kBtu)", "PropertyGFATotal", "DataYear"]].dtypes)

SiteEnergyUse(kBtu)    float64
PropertyGFATotal       float64
DataYear                 int64
dtype: object


In [82]:
print(df_all.groupby("DataYear")["ComplianceStatus"].value_counts(dropna=False))

DataYear  ComplianceStatus            
2015      Compliant                       3156
          Not Compliant                     19
2016      Compliant                       3240
          Error - Correct Default Data     113
          Non-Compliant                     37
          Not Compliant                     30
          Missing Data                      15
2017      Compliant                       3049
          Not Compliant                    261
2018      Compliant                       2997
          Not Compliant                    384
2019      Compliant                       3179
          Not Compliant                    277
2020      Compliant                       3364
          Not Compliant                    166
2021      Compliant                       3387
          Not Compliant                    191
2022      Compliant                       3113
          Not Compliant                    504
2023      Compliant                       3544
          Not Complia

In [83]:
print(df_all["ComplianceStatus"].value_counts(dropna=False))

ComplianceStatus
Compliant                       32604
Not Compliant                    2112
Error - Correct Default Data      113
Non-Compliant                      37
Missing Data                       15
Name: count, dtype: int64


In [84]:
df_hist_model = df_hist_model[df_hist_model["ComplianceStatus"] == "Compliant"].copy()